# Wildfire Backbone Benchmark — Kaggle Runner

**Before running:** open the notebook Settings panel (right sidebar) and set:
- **Accelerator** → GPU T4 x2 (or P100)
- **Internet** → On (needed to `pip install timm` and download the dataset/checkpoints from GitHub)

This notebook is fully self-contained — it writes out the exact same project code used in the
`wildfire-backbone-benchmark` repo (stratified split → train 3 backbones → evaluate with
hard-example mining → compare → annotate a demo video), so results are apples-to-apples with any
CPU run of the same code. Run all cells top to bottom, then copy the final summary table/plots
back to report the numbers.

In [ ]:
!pip install -q timm opencv-python-headless
import torch
print('cuda available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 1. Download the dataset

Same source used in the reference CPU run: [DeepQuestAI/Fire-Smoke-Dataset](https://github.com/DeepQuestAI/Fire-Smoke-Dataset)
(3,000 real photos, `Fire` / `Neutral` / `Smoke`, no login needed), mapped onto
`fire` / `no_fire` / `start_fire` (smoke-only = the same "early onset" idea as `start_fire`).

In [ ]:
%cd /kaggle/working
!wget -q -O fire_smoke.zip https://github.com/DeepQuestAI/Fire-Smoke-Dataset/releases/download/v1/FIRE-SMOKE-DATASET.zip
!unzip -q fire_smoke.zip -d fire_smoke_data
!mkdir -p data/raw/fire data/raw/no_fire data/raw/start_fire
import shutil, os
for split in ['Train', 'Test']:
    for src_cls, dst_cls in [('Fire', 'fire'), ('Neutral', 'no_fire'), ('Smoke', 'start_fire')]:
        src_dir = f'fire_smoke_data/FIRE-SMOKE-DATASET/{split}/{src_cls}'
        for fname in os.listdir(src_dir):
            shutil.copy(f'{src_dir}/{fname}', f'data/raw/{dst_cls}/{split.lower()}_{fname}')
for c in ['fire', 'no_fire', 'start_fire']:
    print(c, len(os.listdir(f'data/raw/{c}')))

## 2. Write out the project code

In [ ]:
!mkdir -p data models results/checkpoints results/metrics results/hard_examples results/plots results/videos
!touch data/__init__.py models/__init__.py

In [ ]:
%%writefile data/dataset.py
import json

from PIL import Image
from torch.utils.data import Dataset


class FireDataset(Dataset):
    def __init__(self, split_name, transform, split_file="data/split.json"):
        with open(split_file) as f:
            data = json.load(f)[split_name]
        self.samples = data
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        return self.transform(img), label

In [ ]:
%%writefile data/prepare_split.py
import argparse
import json
import os

from sklearn.model_selection import train_test_split

CLASS_NAMES = ["fire", "no_fire", "start_fire"]


def build_split(data_dir, out_path, seed=42):
    samples = []
    for label, cls in enumerate(CLASS_NAMES):
        cls_dir = os.path.join(data_dir, cls)
        for fname in sorted(os.listdir(cls_dir)):
            samples.append((os.path.join(cls_dir, fname), label))

    paths, labels = zip(*samples)
    train_p, temp_p, train_l, temp_l = train_test_split(
        paths, labels, test_size=0.3, stratify=labels, random_state=seed)
    val_p, test_p, val_l, test_l = train_test_split(
        temp_p, temp_l, test_size=0.5, stratify=temp_l, random_state=seed)

    split = {
        "train": list(zip(train_p, train_l)),
        "val": list(zip(val_p, val_l)),
        "test": list(zip(test_p, test_l)),
    }
    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
    with open(out_path, "w") as f:
        json.dump(split, f)
    print({k: len(v) for k, v in split.items()})


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--data_dir", default="data/raw")
    parser.add_argument("--out", default="data/split.json")
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()
    build_split(args.data_dir, args.out, args.seed)

In [ ]:
%%writefile models/build_model.py
import timm


def build_model(backbone_name, num_classes=3, freeze_backbone=True):
    model = timm.create_model(backbone_name, pretrained=True, num_classes=num_classes)

    if freeze_backbone:
        for name, param in model.named_parameters():
            if "head" not in name and "fc" not in name and "classifier" not in name:
                param.requires_grad = False

    return model

In [ ]:
%%writefile train.py
import argparse
import time

import timm
import torch
from torch.utils.data import DataLoader

from data.dataset import FireDataset
from models.build_model import build_model


def train(backbone_name, epochs=6, batch_size=32, lr=1e-3, num_workers=2):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"[{backbone_name}] device={device}")
    model = build_model(backbone_name).to(device)

    cfg = timm.data.resolve_data_config({}, model=model)
    transform = timm.data.create_transform(**cfg, is_training=True)
    val_transform = timm.data.create_transform(**cfg, is_training=False)

    train_ds = FireDataset("train", transform)
    val_ds = FireDataset("val", val_transform)
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers)
    val_dl = DataLoader(val_ds, batch_size=batch_size, num_workers=num_workers)

    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    criterion = torch.nn.CrossEntropyLoss()
    use_amp = device == "cuda"
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    best_val_acc = 0
    for epoch in range(epochs):
        t0 = time.time()
        model.train()
        total_loss = 0.0
        for imgs, labels in train_dl:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=use_amp):
                out = model(imgs)
                loss = criterion(out, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item() * imgs.size(0)

        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for imgs, labels in val_dl:
                imgs, labels = imgs.to(device), labels.to(device)
                preds = model(imgs).argmax(1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)
        val_acc = correct / total
        dt = time.time() - t0
        print(f"[{backbone_name}] epoch {epoch}: train_loss={total_loss/len(train_ds):.4f} "
              f"val_acc={val_acc:.4f} ({dt:.1f}s)")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), f"results/checkpoints/{backbone_name}.pt")
            print(f"[{backbone_name}] saved new best checkpoint (val_acc={val_acc:.4f})")

    print(f"[{backbone_name}] done. best_val_acc={best_val_acc:.4f}")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--backbone", required=True)
    parser.add_argument("--epochs", type=int, default=6)
    parser.add_argument("--batch_size", type=int, default=32)
    parser.add_argument("--lr", type=float, default=1e-3)
    parser.add_argument("--num_workers", type=int, default=2)
    args = parser.parse_args()
    train(args.backbone, epochs=args.epochs, batch_size=args.batch_size,
          lr=args.lr, num_workers=args.num_workers)

In [ ]:
%%writefile evaluate.py
import json
import os
import shutil
import time

import timm
import torch
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader

from data.dataset import FireDataset
from models.build_model import build_model

CLASS_NAMES = ["fire", "no_fire", "start_fire"]


def evaluate(backbone_name, hard_example_conf_threshold=0.5, latency_runs=50):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = build_model(backbone_name)
    model.load_state_dict(torch.load(f"results/checkpoints/{backbone_name}.pt", map_location=device))
    model.to(device).eval()

    cfg = timm.data.resolve_data_config({}, model=model)
    transform = timm.data.create_transform(**cfg, is_training=False)
    test_ds = FireDataset("test", transform)
    test_dl = DataLoader(test_ds, batch_size=32, shuffle=False)

    hard_dir = f"results/hard_examples/{backbone_name}"
    if os.path.exists(hard_dir):
        shutil.rmtree(hard_dir)
    for cls in CLASS_NAMES:
        os.makedirs(f"{hard_dir}/{cls}", exist_ok=True)

    all_preds, all_labels = [], []
    sample_idx = 0
    with torch.no_grad():
        for imgs, labels in test_dl:
            imgs = imgs.to(device)
            logits = model(imgs)
            probs = torch.softmax(logits, dim=1).cpu()
            preds = probs.argmax(1)

            for i in range(len(labels)):
                true_label = labels[i].item()
                pred_label = preds[i].item()
                confidence_for_true_class = probs[i, true_label].item()

                if pred_label != true_label or confidence_for_true_class < hard_example_conf_threshold:
                    src_path, _ = test_ds.samples[sample_idx]
                    fname = os.path.basename(src_path)
                    true_cls = CLASS_NAMES[true_label]
                    pred_cls = CLASS_NAMES[pred_label]
                    dst_name = f"pred-{pred_cls}_conf-{confidence_for_true_class:.2f}_{fname}"
                    shutil.copy(src_path, f"{hard_dir}/{true_cls}/{dst_name}")

                sample_idx += 1

            all_preds.extend(preds.tolist())
            all_labels.extend(labels.tolist())

    report = classification_report(all_labels, all_preds,
                                    target_names=CLASS_NAMES, output_dict=True)
    cm = confusion_matrix(all_labels, all_preds).tolist()

    model.eval()
    dummy = torch.randn(1, 3, cfg["input_size"][1], cfg["input_size"][2]).to(device)
    times = []
    for _ in range(latency_runs):
        t0 = time.time()
        with torch.no_grad():
            model(dummy)
        times.append(time.time() - t0)
    latency_ms = sum(times) / len(times) * 1000

    n_params = sum(p.numel() for p in model.parameters())
    n_hard_examples = sum(len(os.listdir(f"{hard_dir}/{c}")) for c in CLASS_NAMES)

    result = {"model": backbone_name, "report": report, "confusion_matrix": cm,
              "latency_ms": latency_ms, "params": n_params,
              "n_hard_examples": n_hard_examples}
    os.makedirs("results/metrics", exist_ok=True)
    with open(f"results/metrics/{backbone_name}.json", "w") as f:
        json.dump(result, f, indent=2)
    print(backbone_name, "acc:", report["accuracy"], "latency:", latency_ms,
          "ms | hard examples saved:", n_hard_examples)


if __name__ == "__main__":
    for name in ["resnet50", "efficientnet_b0", "vit_tiny_patch16_224"]:
        evaluate(name)

In [ ]:
%%writefile compare_results.py
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

BACKBONES = ["resnet50", "efficientnet_b0", "vit_tiny_patch16_224"]
DISPLAY_NAMES = {"resnet50": "ResNet50", "efficientnet_b0": "EfficientNet-B0",
                  "vit_tiny_patch16_224": "ViT-Tiny"}
CLASS_NAMES = ["fire", "no_fire", "start_fire"]


def load_results():
    rows = []
    for name in BACKBONES:
        with open(f"results/metrics/{name}.json") as f:
            r = json.load(f)
        rows.append({
            "Model": DISPLAY_NAMES[name],
            "Test Acc": r["report"]["accuracy"],
            "F1 (start_fire)": r["report"]["start_fire"]["f1-score"],
            "Latency (ms)": r["latency_ms"],
            "Params (M)": r["params"] / 1e6,
            "Hard Examples": r["n_hard_examples"],
        })
    return pd.DataFrame(rows)


def plot_comparison(df, out_path="results/plots/comparison.png"):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].bar(df["Model"], df["Test Acc"], color="#3b82f6")
    axes[0].set_title("Test Accuracy")
    axes[0].set_ylim(0, 1)

    axes[1].bar(df["Model"], df["Latency (ms)"], color="#f97316")
    axes[1].set_title("Latency (ms/image, GPU)")

    axes[2].bar(df["Model"], df["Params (M)"], color="#10b981")
    axes[2].set_title("Params (M)")

    for ax in axes:
        ax.tick_params(axis="x", rotation=20)

    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    print(f"wrote {out_path}")


def plot_confusion_matrices():
    fig, axes = plt.subplots(1, len(BACKBONES), figsize=(5 * len(BACKBONES), 4.5))
    for ax, name in zip(axes, BACKBONES):
        with open(f"results/metrics/{name}.json") as f:
            r = json.load(f)
        cm = np.array(r["confusion_matrix"])
        cm_norm = cm / cm.sum(axis=1, keepdims=True)

        im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
        ax.set_xticks(range(len(CLASS_NAMES)))
        ax.set_yticks(range(len(CLASS_NAMES)))
        ax.set_xticklabels(CLASS_NAMES, rotation=30, ha="right")
        ax.set_yticklabels(CLASS_NAMES)
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True")
        ax.set_title(DISPLAY_NAMES[name])
        for i in range(len(CLASS_NAMES)):
            for j in range(len(CLASS_NAMES)):
                ax.text(j, i, f"{cm[i, j]}\n({cm_norm[i, j]:.0%})",
                        ha="center", va="center",
                        color="white" if cm_norm[i, j] > 0.5 else "black", fontsize=9)
    fig.tight_layout()
    fig.savefig("results/plots/confusion_matrices.png", dpi=150)
    print("wrote results/plots/confusion_matrices.png")


if __name__ == "__main__":
    df = load_results()
    print(df.to_string(index=False))
    df.to_csv("results/metrics/comparison.csv", index=False)
    plot_comparison(df)
    plot_confusion_matrices()

In [ ]:
%%writefile infer_video.py
import argparse
import os

import cv2
import timm
import torch
from PIL import Image

from models.build_model import build_model

CLASS_NAMES = ["fire", "no_fire", "start_fire"]
BOX_COLOR = {"fire": (0, 0, 255), "no_fire": (0, 200, 0), "start_fire": (0, 165, 255)}


def annotate_video(backbone_name, input_path, output_path, freq=12):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = build_model(backbone_name)
    model.load_state_dict(torch.load(f"results/checkpoints/{backbone_name}.pt", map_location=device))
    model.to(device).eval()

    cfg = timm.data.resolve_data_config({}, model=model)
    transform = timm.data.create_transform(**cfg, is_training=False)

    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        raise RuntimeError(f"could not open {input_path}")

    fps = cap.get(cv2.CAP_PROP_FPS) or 24
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    os.makedirs(os.path.dirname(output_path) or ".", exist_ok=True)
    writer = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

    last_label, last_conf = "unknown", 0.0
    frame_idx = 0

    while True:
        ok, frame = cap.read()
        if not ok:
            break

        if frame_idx % freq == 0:
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_img = Image.fromarray(rgb)
            tensor = transform(pil_img).unsqueeze(0).to(device)
            with torch.no_grad():
                probs = torch.softmax(model(tensor), dim=1)[0].cpu()
            pred_idx = probs.argmax().item()
            last_label, last_conf = CLASS_NAMES[pred_idx], probs[pred_idx].item()

        color = BOX_COLOR.get(last_label, (255, 255, 255))
        text = f"{last_label}: {last_conf*100:.1f}%"
        (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 1, 2)
        cv2.rectangle(frame, (10, 10), (20 + tw, 40 + th), (0, 0, 0), cv2.FILLED)
        cv2.putText(frame, text, (15, 35 + th), cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)

        writer.write(frame)
        frame_idx += 1

    cap.release()
    writer.release()
    print(f"wrote {output_path} ({frame_idx} frames, backbone={backbone_name}, freq={freq})")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--backbone", required=True)
    parser.add_argument("--input", required=True)
    parser.add_argument("--output", default="results/videos/annotated.mp4")
    parser.add_argument("--freq", type=int, default=12)
    args = parser.parse_args()
    annotate_video(args.backbone, args.input, args.output, args.freq)

## 3. Build the split (stratified 70/15/15, seed 42 — same split logic as the CPU run)

In [ ]:
!python data/prepare_split.py --data_dir data/raw --out data/split.json

## 4. Train all three backbones (frozen-backbone linear probe, 6 epochs each)

On a T4 this should take a few minutes total instead of ~40 on CPU.

In [ ]:
!python train.py --backbone resnet50 --epochs 6
!python train.py --backbone efficientnet_b0 --epochs 6
!python train.py --backbone vit_tiny_patch16_224 --epochs 6

## 5. Evaluate all three (metrics, confusion matrix, latency, hard-example mining)

In [ ]:
!python evaluate.py

## 6. Compare — this is the table + plots to report back

In [ ]:
!python compare_results.py
import pandas as pd
df = pd.read_csv('results/metrics/comparison.csv')
print(df.to_markdown(index=False))
from IPython.display import Image as IPImage, display
display(IPImage('results/plots/comparison.png'))
display(IPImage('results/plots/confusion_matrices.png'))

## 7. (Optional) Video demo with the winning backbone

Swap `--backbone` for whichever model wins the comparison above.

In [ ]:
!wget -q -O demo_input.mp4 https://raw.githubusercontent.com/spacewalk01/yolov5-fire-detection/master/input.mp4
!python infer_video.py --backbone resnet50 --input demo_input.mp4 --output results/videos/annotated.mp4

## 8. Zip everything worth reporting back

In [ ]:
!zip -qr wildfire_benchmark_results.zip results/metrics results/plots results/videos
print('Zipped -> /kaggle/working/wildfire_benchmark_results.zip (download from the notebook Output tab)')
print()
print('=== COPY THE TABLE ABOVE (step 6) BACK TO REPORT METRICS ===')